In [12]:
import pandas as pd
import numpy as np
import re
import string
from nltk.stem import WordNetLemmatizer

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report,f1_score

import mlflow
import mlflow.sklearn

In [2]:
df=pd.read_csv('../data/ticket_classification.csv')
df=df.dropna()

In [3]:
df.head()

,subject,body,answer,type,queue,priority
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high


In [4]:
def clean_text(text):
    lemmatizer= WordNetLemmatizer()
    stop_words=set(stopwords.words('english'))
    # Lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords + lemmatization
    cleaned_words = []

    for word in words:

        # Remove stopwords
        if word not in stop_words:

            # Lemmatize
            lemma_word = lemmatizer.lemmatize(word)

            cleaned_words.append(lemma_word)

    # Join back
    text = " ".join(cleaned_words)

    return text

In [5]:
df['Full Text'] = df['subject'].apply(clean_text)+" "+df['body'].apply(clean_text)

# Let's see what a row looks like before and after
print("BEFORE:\n", df['body'].iloc[0])
print("\nAFTER:\n", df['Full Text'].iloc[0])

BEFORE:
 Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in multiple times using different browsers and devices, but the issue persists.\n\nCould you please provide an update on the outage status and an estimated time for resolution? Also, are there any alternative ways to access and manage my account during this downtime?

AFTER:
 account disruption dear customer support teamnni writing report significant problem centralized account management portal currently appears offline outage blocking access account setting leading substantial inconvenience attempted log multiple time using different browser device issue persistsnncould please provide update outage status estimated time resolution also alternative way access manage account downtime


In [6]:
from sklearn.preprocessing import LabelEncoder
target_encoder = LabelEncoder()

# Fit and transform your 'ticket type' column into numerical integers
df['Target_Encoded'] = target_encoder.fit_transform(df['type'])

# Let's see the mapping to understand what it did
mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))
print("Category Mapping Reference:")
print(mapping)

Category Mapping Reference:
{'Change': 0, 'Incident': 1, 'Problem': 2, 'Request': 3}


In [7]:
X = df['Full Text']
y = df['Target_Encoded']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
random_state=42,stratify=y)

In [9]:
tfidf = TfidfVectorizer(max_features=10000,ngram_range=(1,2),
                        min_df=2,
                        max_df=0.95)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [10]:
X_train_tfidf=X_train_tfidf.astype('float32', copy=False)
X_test_tfidf=X_test_tfidf.astype('float32', copy=False)

In [13]:
log=LogisticRegression(
        max_iter=2000,
        class_weight='balanced'
    )
log.fit(X_train_tfidf,y_train)
predictions=log.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, predictions)
f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")
print(f"F1-Score (Macro): {f1_macro:.4f}\n")

Accuracy: 0.8398
F1-Score (Weighted): 0.8421
F1-Score (Macro): 0.8522



In [15]:
from xgboost import XGBClassifier
objective='multi:softprob'
eval_metric='mlogloss'
random_state=42
max_depth=10
n_jobs=-1
n_estimators=100
xgb_model = XGBClassifier(
    objective=objective,
    eval_metric=eval_metric,
    random_state=random_state,
    max_depth=max_depth,
    n_jobs=-1,
    n_estimators=n_estimators
      )
xgb_model.fit(X_train_tfidf, y_train)

predictions = xgb_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, predictions)
f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
print("Accuracy:",
            accuracy_score(y_test, predictions))

print("F1 Score:",
    f1_score(y_test, predictions, average='macro'))

print(classification_report(y_test, predictions))

Accuracy: 0.8732737611697806
F1 Score: 0.8670313205655684
              precision    recall  f1-score   support

           0       0.97      0.91      0.94       511
           1       0.81      0.93      0.86      1957
           2       0.82      0.59      0.68      1028
           3       0.97      0.99      0.98      1428

    accuracy                           0.87      4924
   macro avg       0.89      0.85      0.87      4924
weighted avg       0.87      0.87      0.87      4924

